In [0]:
from pyspark.sql.functions import *
from delta import DeltaTable

### Create Flag Parameter

In [0]:
dbutils.widgets.text("incemental_flag", '0')
incemental_flag = dbutils.widgets.get("incemental_flag")
print(incemental_flag)

### Create Fact _Tables_

Read silver table

In [0]:
silver_df = spark.read.format("parquet")\
    .load("/Volumes/adbkomal/silver/silver_data/CarSalesData")

display(silver_df)

Read all DIMENSIONS

In [0]:
dealer_df = spark.sql("""
                      SELECT * FROM adbkomal.gold.dim_dealer
                      """)
branch_df = spark.sql("""
                      SELECT * FROM adbkomal.gold.dim_branch
                      """)
model_df = spark.sql("""
                      SELECT * FROM adbkomal.gold.dim_model
                      """)
date_df = spark.sql("""
                      SELECT * FROM adbkomal.gold.dim_date
                      """)

In [0]:
fact_df = silver_df.join(dealer_df, silver_df["Dealer_ID"] == dealer_df["Dealer_ID"], "left")\
                    .join(branch_df, silver_df["Branch_ID"] == branch_df["Branch_ID"], "left")\
                    .join(model_df, silver_df["Model_ID"] == model_df["Model_ID"], "left")\
                    .join(date_df, silver_df["Date_ID"] == date_df["Date_ID"], "left")\
                    .select(silver_df["Revenue"], silver_df["Units_Sold"], silver_df["revenue_per_unit"], dealer_df["dim_dealer_key"], branch_df["dim_branch_key"], model_df["dim_model_key"], date_df["dim_date_key"])


In [0]:
display(fact_df)

### Slowly Changing Dimension - Type 1 (Upsert)

In [0]:
if spark.catalog.tableExists("adbkomal.gold.fact_car_sales"):
    print("incremental run")
    delta_table = DeltaTable.forPath(spark, "abfss://gold@adfstgkomal.dfs.core.windows.net/fact_car_sales")
    delta_table.alias("target")\
        .merge(fact_df.alias("src"), "target.dim_branch_key = src.dim_branch_key and target.dim_dealer_key = src.dim_dealer_key and target.dim_model_key = src.dim_model_key and target.dim_date_key = src.dim_date_key" )\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    print("merge completed")
else:
    print("Initial run")
    fact_df.write.format("delta")\
        .mode("overwrite")\
        .option("path","abfss://gold@adfstgkomal.dfs.core.windows.net/fact_car_sales")\
        .saveAsTable("adbkomal.gold.fact_car_sales")
    


In [0]:
%sql
Select * from adbkomal.gold.fact_car_sales;